In [ ]:
"""
Implemente o Algoritmo K-Nearest Neighbors (KNN) do Zero
"""

import numpy as np
from collections import Counter
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import json
import os
from PIL import Image

class KNN_do_Zero:
    """
    Implementação do algoritmo K-Nearest Neighbors do zero
    """

    def __init__(self, k=3):
        """
        Inicializa o classificador KNN

        Parâmetros:
        -----------
        k : int
            Número de vizinhos mais próximos a considerar
        """
        self.k = k

    def calcular_distancia(self, ponto1, ponto2):
        """
        Calcula a distância euclidiana
        """
        return np.sqrt(np.sum((ponto1-ponto2)**2))


    def fit(self, X_train, y_train):
        """
        'Treina' o modelo (apenas armazena os dados de treino)

        No KNN não há treinamento real, apenas memorização dos dados
        """
        self.X_train = np.array(X_train)
        self.y_train = np.array(y_train)

    def predict(self, X_test):
        """
        Prediz as classes para múltiplas amostras
        """
        previsoes = []

        for x in X_test:
            # 1. Calcular as distâncias entre o ponto 'x' atual e todos os pontos de treino
            distancias = [self.calcular_distancia(x, x_treino) for x_treino in self.X_train]

            # 2. Pegar os índices dos 'k' vizinhos mais próximos
            k_indices = np.argsort(distancias)[:self.k]

            # 3. Pegar as classes (nomes) desses 'k' vizinhos
            k_vizinhos_labels = [self.y_train[i] for i in k_indices]

            # 4. Obter a classe mais comum e adicionar à lista de previsões
            mais_comum = Counter(k_vizinhos_labels).most_common(1)
            previsoes.append(mais_comum[0][0])
        return np.array(previsoes)
    def score(self, X_test, y_test):
        """
        Calcula a acurácia do modelo
        """
        previsoes = self.predict(X_test)
        return np.sum(previsoes == y_test) / len(y_test)



In [ ]:
caminho_arquivo = "embeddings_mestre.json"

#carrega os dados
with open(caminho_arquivo, "r", encoding = "utf-8") as f:
  dados = json.load(f)
#prepara a lista com caracteristicas x e rotulo y
X_rostos = []
y_rostos = []
caminho_rostos = []

for item in dados:
  X_rostos.append(item["embedding"])
  y_rostos.append(item["nome"])
  caminho_rostos.append(item.get("caminho_recorte", "Recorte Desconhecido"))

#converte para array do numpy pra facilitar os calculos
X_rostos = np.array(X_rostos)
y_rostos = np.array(y_rostos)
X_rostos = X_rostos/np.linalg.norm(X_rostos, axis=1, keepdims=True)
caminho_rostos = np.array(caminho_rostos)

print(f"Total de rostos carregados: {len(y_rostos)}")
print(f"Tamanho de cada embedding (features): {len(X_rostos[0])}")

#usando o 20% de fotos para testar se o modelo aprendeu
X_train_r, X_test_r, y_train_r, y_test_r, caminhos_train, caminhos_test = train_test_split(
    X_rostos, y_rostos, caminho_rostos, test_size=0.2, random_state=42
)

knn = KNN_do_Zero(k=3)
knn.fit(X_train_r, y_train_r)
previsoes = knn.predict(X_test_r)
acuracia_train=knn.score(X_train_r, y_train_r)
acuracia_test=knn.score(X_test_r, y_test_r)

print(f"\nAcurácia do Reconhecimento Facial no teste: {acuracia_test*100:.2f}%")
print(f"\nAcurácia do Reconhecimento Facial no treino: {acuracia_train*100:.2f}%")



In [ ]:
# =============================================================================
# BLOCO 3: EXIBIÇÃO VISUAL DAS IMAGENS E RESULTADOS DO TESTE
# =============================================================================

num_fotos = len(y_test_r)
cols = 4  # Número de imagens exibidas por linha
rows = (num_fotos + cols - 1) // cols

plt.figure(figsize=(15, 3.5 * rows))

for i, (caminho, real, previsto) in enumerate(zip(caminhos_test, y_test_r, previsoes)):
    plt.subplot(rows, cols, i + 1)

    # 1. Carrega e desenha a foto recortada se o arquivo existir no disco
    if os.path.exists(caminho):
        img = Image.open(caminho)
        plt.imshow(img)
    else:
        # Mostra aviso caso o caminho guardado no JSON não seja localizado
        nome_arquivo = os.path.basename(caminho)
        plt.text(0.5, 0.5, f"Imagem não encontrada:\n{nome_arquivo}",
                 ha='center', va='center', fontsize=8, color='gray')
        plt.xlim(0, 1)
        plt.ylim(0, 1)

    # 2. Avalia acerto/erro para definir status e cor do título
    acertou = (real == previsto)
    cor = "green" if acertou else "red"
    # Em vez de emojis, use texto padronizado:
    status = "[OK]" if acertou else "[ERRO]"

    # 3. Monta o cabeçalho de cada imagem
    plt.title(f"Real: {real}\nIA: {previsto} ({status})",
              color=cor, fontsize=10, fontweight='bold')
    plt.axis('off')

plt.tight_layout()
plt.show()